Cell 1 — Title
markdown
# Project 1 — Completion Layer
## Filling the gaps between the blueprint(s) and the PDF Project 1 specification

**Purpose of this notebook:**  
The existing blueprints (end-to-end notebook, E-Commerce Risk & Demand Intelligence System, AI/ML Engineering Master Blueprint) describe the *engineering pattern* and the *metrics contract*, but they do not contain: SQL feature extraction, churn definition, calibration, MLflow tracking, a deliberate failure postmortem, drift monitoring, a rollback procedure, or a reproducibility manifest.

This notebook is where those missing pieces get built — for **order cancellation risk** (or a churn variant if you decide to reframe).

**Rules:**
- Every section below is a task. Write the code yourself.
- Do not skip a section because it "seems optional." The PDF requires all of them.
- Record evidence (file path, screenshot, query output, metric value) after each task.
- Keep a decision log at the bottom.

**Where each section belongs:** stated at the top of each section, relative to the original end-to-end blueprint (Parts 0–25).
Cell 2 — Gap map
markdown
## Gap Map: What the blueprints cover vs what the PDF requires

| PDF Project 1 requirement | Covered by blueprints? | Where it must be added |
|---|---|---|
| Business problem definition | Partially | Section 1 below |
| Churn / cancellation definition | ✘ | Section 1 |
| Prediction horizon | ✘ | Section 1 |
| Prediction-time feature availability | ✔ (blueprint 1 Part 8, blueprint 2 §5) | — |
| Leakage audit | ✔ | — |
| Naive baseline | ✔ | — |
| At least 3 candidate models | ✘ (only 2) | Section 4 |
| Cross-validation | ✔ | — |
| Calibration | ✘ | Section 5 |
| Threshold study | ✔ | — |
| Cost-driven threshold with real costs | ✘ (only conceptual) | Section 11 |
| Error slicing by group | ✘ | Section 7 |
| Deliberate failure postmortem | ✘ | Section 9 |
| Temporal split comparison | ✘ | Section 6 |
| PostgreSQL schema | ✔ (you already built it) | — |
| **SQL feature extraction** | ✘ | Section 3 |
| Experiment tracking (MLflow) | ✘ | Section 8 |
| FastAPI | Described, not implemented | Section 12 |
| PostgreSQL prediction logging | Described, not implemented | Section 12 |
| Tests | Described, not implemented | Section 13 |
| Docker | Described, not implemented | Section 13 |
| CI/CD | Described, not implemented | Section 14 |
| Deployment | Described, not implemented | Section 15 |
| Drift monitoring design | ✘ | Section 16 |
| Rollback procedure | ✘ | Section 17 |
| Reproducibility manifest | ✘ | Section 10 |
| API performance metrics | ✘ | Section 18 |
| Baseline recording card | Conceptually in blueprint 3 | Section 2 |
| Final project scorecard | Conceptually in blueprint 3 §11 | Section 20 |
Cell 3 — Section 1: Prediction contract
markdown
## Section 1 — Prediction Contract
**Where this belongs:** replaces the loose framing in blueprint 1, Part 8 ("The exact prediction contract").

**Why:** The PDF requires an explicit prediction moment and horizon. A vague target produces a vague model.

**Task 1.1 — Define the target**
Write, in plain English:
- What exactly is the positive class?
- What exactly is the negative class?
- What event triggers a prediction?
- What outcome are we trying to anticipate?
- What is the time window between prediction and outcome?

**Task 1.2 — Define the prediction horizon**
If predicting cancellation:
- Is cancellation defined by `order_status == "Cancelled"`, or by a stricter rule?
- Is there a time limit (e.g., cancelled within 7 days of order creation)?

If predicting churn:
- Define "churned" — e.g., *"a customer is churned if they make no purchase in the 90 days following the snapshot date."*
- Choose the snapshot date logic.
- Justify why 90 days (or your chosen window) is the right horizon.

**Task 1.3 — State who acts on the output**
- Who receives the prediction?
- What action do they take for a high-risk prediction?
- What action do they take for a low-risk prediction?

**Task 1.4 — State the cost of each mistake**
- Cost of a false positive (in dollars, hours, or both).
- Cost of a false negative.
- Document the source of these numbers (estimate, interview, assumption).

**Deliverable:** a written markdown contract at the top of the notebook, before any code.
Cell 4 — Section 2: Baseline recording card
markdown
## Section 2 — Baseline Recording Card
**Where this belongs:** before Part 8 of blueprint 1, or as a standalone artifact.

**Why:** Blueprint 3 §4 requires a recorded baseline before any optimization. Without it, "improvement" is unprovable.

**Task 2.1 — Create the baseline card**
Fill in every field as a markdown table:

| Field | Value |
|---|---|
| System / model / version | |
| Dataset (name, size, version/hash) | |
| Hardware (CPU/GPU, RAM) | |
| Train / test split | |
| Baseline model | |
| Primary metric | |
| Baseline metric value | |
| P95 inference latency | |
| P99 inference latency | |
| Throughput (RPS) | |
| Error rate | |
| Cost per 1,000 predictions | |
| Date recorded | |

**Task 2.2 — State the acceptance threshold**
Write a single sentence:
> *"The model is accepted only if [primary metric] ≥ [value] AND P95 latency ≤ [ms] AND cost/1k ≤ [$]."*

**Deliverable:** one markdown table plus one acceptance sentence. This becomes the contract for the rest of the project.
Cell 5 — Section 3: SQL feature extraction
markdown
## Section 3 — SQL Feature Extraction (the biggest gap)
**Where this belongs:** immediately before Part 8 of blueprint 1, replacing the in-pandas feature build.

**Why:** The PDF Section 15.3 requires *SQL feature extraction*. All three blueprints do feature engineering in pandas only. That is not acceptable for Project 1.

You already created the e-commerce database — this section is about **pulling features out of it with SQL**.

**Task 3.1 — Write the base feature query**
Using `psycopg2` or `SQLAlchemy`, connect Python to PostgreSQL and pull one row per prediction unit (per order for cancellation; per customer per snapshot for churn).

The query must include:
- A `SELECT` with explicit column names (no `SELECT *`).
- At least one `JOIN` between two tables.
- At least one `GROUP BY` with an aggregate.
- At least one `CASE WHEN` for a derived flag.
- At least one `COALESCE` or `NULLIF` to handle nulls at source.
- A `WHERE` clause that excludes rows that would not exist at prediction time.

**Task 3.2 — Add a window function**
Include at least one of:
- `ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date)` — to compute previous-order count.
- `LAG(...) OVER (...)` — to compute days since previous order.
- `SUM(...) OVER (PARTITION BY customer_id ORDER BY order_date ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING)` — to compute prior cumulative spend *without* leaking the current order.

**Task 3.3 — Add a CTE**
Wrap the base query in at least one `WITH` clause so the query is readable and can be reused.

**Task 3.4 — Verify with EXPLAIN**
Run `EXPLAIN ANALYZE` on the query. Save the output. Identify:
- Whether indexes are used.
- Whether any sequential scans appear on large tables.
- Whether you should add an index on `orders(order_date)` or `orders(customer_id)`.

**Task 3.5 — Leakage audit at the SQL layer**
For every column in the final query, write one line:
> *"This column is safe because it is [known at order creation / computed only from prior rows]."*

If any column cannot be justified, remove it.

**Task 3.6 — Save the query**
Save the final SQL as `src/sql/cancellation_features.sql` (or `churn_features.sql`). Do not keep it only in the notebook.

**Deliverable:**
- One `.sql` file in `src/sql/`.
- A markdown table listing every feature and its prediction-time justification.
- A screenshot or text capture of `EXPLAIN ANALYZE`.
Cell 6 — Section 4: Third candidate model
markdown
## Section 4 — Third Candidate Model
**Where this belongs:** Part 8 of blueprint 1, after Logistic Regression and Random Forest.

**Why:** The PDF Section 15.3 requires *at least three candidate models*. The blueprint has two.

**Task 4.1 — Add a gradient boosting model**
Add one of:
- `HistGradientBoostingClassifier` (fast, handles categoricals poorly — needs encoding), or
- `GradientBoostingClassifier` (slower but well-understood), or
- `XGBoost` / `LightGBM` if available.

Train it inside the same `Pipeline` structure as the other two.

**Task 4.2 — Compare all three**
Build a table with columns:
- Model name
- CV mean F1
- CV std F1
- Test accuracy
- Test precision
- Test recall
- Test F1
- Test ROC-AUC
- Test PR-AUC

**Task 4.3 — Justify the winner**
Write one paragraph explaining which model you chose and why. Do not choose the most complex model by default.

**Deliverable:** a comparison table plus a one-paragraph model selection justification.
Cell 7 — Section 5: Calibration
markdown
## Section 5 — Probability Calibration
**Where this belongs:** after Part 8 of blueprint 1, before Part 9 (threshold engineering).

**Why:** Threshold engineering is meaningless if the probabilities are not calibrated. The PDF Section 14.1 lists calibration as a required classification evaluation item; none of the blueprints address it.

**Task 5.1 — Draw the reliability diagram**
Plot a calibration curve for the best classifier:
- x-axis: predicted probability bins.
- y-axis: actual frequency of the positive class.
- A perfectly calibrated model lies on the diagonal.

**Task 5.2 — Compute the Brier score**
`Brier = mean((p - y)²)`. Lower is better. Compare against the baseline's Brier score.

**Task 5.3 — Apply calibration**
Wrap the best classifier in `CalibratedClassifierCV` using `method="isotonic"` or `method="sigmoid"`. Refit. Recompute the calibration curve and Brier score.

**Task 5.4 — Decide**
Answer:
- Is the raw model calibrated?
- Does calibration change the threshold study results?
- Do you ship the raw model or the calibrated model? Why?

**Deliverable:** calibration curve plot before and after, Brier scores, and a written decision.
Cell 8 — Section 6: Temporal split comparison
markdown
## Section 6 — Temporal Split Comparison
**Where this belongs:** Part 13 of blueprint 1, replacing or supplementing the random split.

**Why:** PDF Section 14.2 warns about temporal leakage. For any prediction involving time, a random split overstates performance.

**Task 6.1 — Build the time-ordered split**
- Sort by `order_date`.
- Train on earlier rows, test on later rows.
- Choose the cutoff date. Justify it.

**Task 6.2 — Compare random vs temporal**
Report the primary metric under both splits. Answer:
- Does random split overstate performance?
- By how much?
- Which split reflects the real deployment scenario?

**Task 6.3 — Time-aware cross-validation**
Use `TimeSeriesSplit` from scikit-learn for CV. Compare the CV mean to the random `StratifiedKFold` mean. Explain the difference.

**Deliverable:** two metric values (random vs temporal), plus a paragraph explaining which is honest.
Cell 9 — Section 7: Error slicing by group
markdown
## Section 7 — Error Slicing by Group
**Where this belongs:** Part 12 of blueprint 1, expanding the error analysis.

**Why:** PDF Section 15.2 asks *"Which groups are poorly served?"* The blueprint only looks at overall error.

**Task 7.1 — Slice errors by segment**
For each of the following, compute precision, recall, and F1 *within the segment*:
- By city
- By category
- By payment method
- By price band
- By discount band
- By order value band

**Task 7.2 — Identify underserved groups**
Flag any segment where:
- Recall is more than 10 percentage points below the overall recall, or
- The segment has fewer than 30 test samples (unstable).

**Task 7.3 — Write the fairness note**
One paragraph:
- Which groups the model serves well.
- Which groups it serves poorly.
- Whether the poor performance is likely due to small sample size or genuine model weakness.
- What action you would recommend before deployment.

**Deliverable:** a segment-level metric table plus a written fairness note.
Cell 10 — Section 8: MLflow experiment tracking
markdown
## Section 8 — Experiment Tracking with MLflow
**Where this belongs:** a new section after Part 8 of blueprint 1, or as a wrapper around Parts 8–13.

**Why:** PDF Section 27.1 requires MLflow for at least one substantial project. The blueprints never mention it.

**Task 8.1 — Set up MLflow**
- Install `mlflow`.
- Start a local tracking server: `mlflow ui`.
- Configure the tracking URI inside the notebook.

**Task 8.2 — Log at least three runs**
For each of Logistic Regression, Random Forest, and the third model, log:
- Parameters: `n_estimators`, `max_depth`, `C`, `class_weight`, etc.
- Metrics: test F1, test ROC-AUC, test PR-AUC, CV mean F1, CV std F1.
- Artifacts: the fitted `joblib` pipeline, the confusion matrix image, the calibration curve.
- Tags: `model_type`, `split_strategy`, `calibration_applied`, `git_commit`.

**Task 8.3 — Register the best model**
Use MLflow Model Registry to register the chosen model. Assign a version number.

**Task 8.4 — Reproduce from MLflow**
Reload the logged model in a fresh cell and confirm predictions match the original.

**Deliverable:** screenshots of the MLflow UI showing the three runs, the metrics, and the registered model version.
Cell 11 — Section 9: Deliberate failure postmortem
markdown
## Section 9 — Deliberate Failure Postmortem
**Where this belongs:** after Parts 8–13 of blueprint 1.

**Why:** PDF Section 15.4 *requires* the learner to deliberately investigate at least one disappointing result. None of the blueprints require this.

**Task 9.1 — Identify a disappointing result**
Pick one of:
- A model that underperformed a baseline on one metric.
- A feature that made things worse.
- A segment where recall collapsed.
- A calibration curve that diverged sharply.
- A CV result whose standard deviation was larger than the improvement.

**Task 9.2 — Write the postmortem**
Use this exact structure (from PDF Section 35):

1. Expected result.
2. Actual result.
3. Evidence.
4. Hypotheses (at least three).
5. Tests performed.
6. Root cause.
7. Corrective action.
8. New result.
9. Remaining uncertainty.

**Task 9.3 — Save the postmortem**
Save as `docs/postmortem_01.md` inside the repo.

**Deliverable:** a written postmortem file, committed to Git, referenced in the README.
Cell 12 — Section 10: Reproducibility manifest
markdown
## Section 10 — Reproducibility Manifest
**Where this belongs:** after Part 15 of blueprint 1 (saving pipelines).

**Why:** PDF Section 27.2 requires code commit, data, environment, seeds, config, model version, and evaluation protocol to be recorded. Blueprint 1 only saves package versions.

**Task 10.1 — Create the manifest**
Write a `models/reproducibility.json` containing:

- `git_commit` — output of `git rev-parse HEAD`.
- `git_branch` — output of `git branch --show-current`.
- `dataset_path`.
- `dataset_sha256` — hash of the CSV or DB snapshot.
- `python_version`.
- `scikit_learn_version`.
- `numpy_version`.
- `pandas_version`.
- `random_seed`.
- `train_test_split_params`.
- `cv_strategy`.
- `model_version` — matching the MLflow registry version.
- `evaluation_protocol` — one sentence.
- `created_at`.

**Task 10.2 — Verify reproducibility**
Clone the repo into a fresh directory, install dependencies from `requirements.txt`, and rerun the training script. Confirm the metric values match within a small tolerance.

**Deliverable:** a `reproducibility.json` file and a screenshot of the reproduced metric.
Cell 13 — Section 11: Cost-driven threshold with real numbers
markdown
## Section 11 — Cost-Driven Threshold with Real Numbers
**Where this belongs:** replaces Part 9 of blueprint 1.

**Why:** Blueprint 1 studies thresholds but never plugs in real costs. Blueprint 2 §9 describes the formula but not the numbers.

**Task 11.1 — Assign dollar values**
Assign:
- `Cost_FP` — cost of manually reviewing an order that would not have been cancelled.
- `Cost_FN` — cost of a cancelled order that was not flagged (refund, logistics, customer trust).

Source these numbers from:
- A reasonable estimate,
- An industry benchmark,
- Or an explicit assumption you write down.

**Task 11.2 — Compute expected cost per threshold**
For thresholds in `[0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70]`, compute:

`ExpectedCost = (FP_count × Cost_FP) + (FN_count × Cost_FN)`

**Task 11.3 — Pick the threshold**
Choose the threshold that minimizes expected cost. Plot expected cost vs threshold.

**Task 11.4 — Sanity-check**
Compare against the F1-optimal threshold. If they differ, explain which one the business would prefer and why.

**Deliverable:** an expected-cost table, a plot, and a written threshold recommendation with justification.
Cell 14 — Section 12: FastAPI + PostgreSQL logging implementation
markdown
## Section 12 — FastAPI and PostgreSQL Logging (implementation, not description)
**Where this belongs:** Part 19 of blueprint 1, but as a built artifact, not a described one.

**Why:** Both the PDF and blueprint 1 describe FastAPI and DB logging. Neither implements them. Project 1 requires a working API and working prediction logging.

**Task 12.1 — Create the FastAPI app**
In `backend/main.py`:
- Load the saved pipeline at startup.
- Define `GET /health`.
- Define `POST /predict/cancellation`.
- Define `POST /predict/quantity`.
- Use Pydantic models in `backend/schemas.py` with field validation.

**Task 12.2 — Define Pydantic validation**
- `quantity > 0`.
- `unit_price > 0`.
- `0 ≤ discount_percent ≤ 100`.
- Categoricals not empty.
- Provide a test that confirms invalid inputs are rejected with HTTP 422.

**Task 12.3 — Create the PostgreSQL logging layer**
In `backend/database.py`:
- Connect using `DATABASE_URL` from environment.
- Create the `prediction_logs` table if not exists.
- Insert a row per prediction with: `prediction_type`, `created_at`, `model_version`, `request_payload`, `prediction_value`, `prediction_probability`, `risk_label`, `status`.

**Task 12.4 — Test end-to-end**
- Start FastAPI locally.
- Send one valid cancellation request with `curl`.
- Send one invalid request.
- Query the database and confirm the row was logged.

**Deliverable:** working `main.py`, `schemas.py`, `database.py`; a `curl` transcript; a database row screenshot.
Cell 15 — Section 13: Tests and Docker
markdown
## Section 13 — Tests and Docker
**Where this belongs:** Part 23 of blueprint 1, implemented.

**Why:** The PDF Section 9.4 and Section 36 require tests and Docker for Project 1. The blueprint only describes them.

**Task 13.1 — Write pytest tests**
In `tests/`:
- `test_features.py` — confirm the SQL feature query returns the expected columns and dtypes.
- `test_validation.py` — confirm Pydantic rejects invalid inputs.
- `test_api.py` — use FastAPI's `TestClient` to hit `/health`, valid `/predict/cancellation`, invalid `/predict/cancellation`.
- `test_model_loading.py` — confirm the saved pipeline loads and predicts.

**Task 13.2 — Dockerfile**
Write a `Dockerfile` that:
- Uses a slim Python base.
- Installs `requirements.txt`.
- Copies `backend/` and `models/`.
- Exposes port 8000.
- Runs `uvicorn main:app --host 0.0.0.0 --port 8000`.

**Task 13.3 — docker-compose.yml**
Define services:
- `backend` — the FastAPI image.
- `db` — PostgreSQL with a named volume.

**Task 13.4 — Verify**
- `docker compose up --build` starts cleanly.
- `curl localhost:8000/health` returns `{"status": "ok"}`.
- A prediction request is logged to the containerised Postgres.

**Deliverable:** passing `pytest` output; a working `docker compose up`; a curl transcript.
Cell 16 — Section 14: CI/CD
markdown
## Section 14 — CI/CD with GitHub Actions
**Where this belongs:** Part 17A of blueprint 1, implemented.

**Why:** The PDF Section 28 requires automated tests, linting, Docker build check, and API smoke test on push. The blueprint only describes this.

**Task 14.1 — Create the workflow**
In `.github/workflows/ci.yml`, define jobs:
- `test` — install deps, run `pytest`.
- `lint` — run `ruff` or `flake8`.
- `build` — `docker build -t ecommerce-risk .`.
- `smoke` — start the container, curl `/health`.

**Task 14.2 — Trigger on push and PR**
Workflow runs on `push` and `pull_request` to `main`.

**Task 14.3 — Verify**
Push a commit that intentionally breaks a test. Confirm the workflow fails. Revert. Confirm it passes.

**Deliverable:** a screenshot of the green workflow run and one of the red run.
Cell 17 — Section 15: Deployment
markdown
## Section 15 — Deployment
**Where this belongs:** Part 22 of blueprint 1, executed.

**Why:** The PDF Section 29 requires at least one major project to be deployed. The blueprint only plans it.

**Task 15.1 — Deploy the backend**
Deploy to Render (or equivalent):
- Set `DATABASE_URL` as an environment variable.
- Set the start command: `uvicorn main:app --host 0.0.0.0 --port $PORT`.
- Confirm `/health` returns OK.

**Task 15.2 — Deploy the database**
Use Neon (or equivalent):
- Create a database.
- Run your schema migration.
- Confirm the backend can write rows.

**Task 15.3 — Deploy the frontend**
Host `frontend/` on a static host. Point the API calls at the deployed backend URL.

**Task 15.4 — Run the end-to-end smoke test**
From the live frontend:
- Submit a valid prediction.
- Confirm the response.
- Confirm a row appears in the Neon database.

**Deliverable:** live backend URL, live frontend URL, and a screenshot of a logged row in Neon.
Cell 18 — Section 16: Drift monitoring design
markdown
## Section 16 — Drift Monitoring Design
**Where this belongs:** a new section after Part 23 of blueprint 1.

**Why:** PDF Sections 27 and 30 require drift and training-serving skew monitoring. No blueprint addresses this.

**Task 16.1 — Define the monitoring plan**
Write a markdown table:

| Signal | What we monitor | Alert threshold | Action |
|---|---|---|---|
| Data drift | Distribution of `unit_price` in requests vs training | PSI > 0.2 | Retrain |
| Data drift | Distribution of `city` in requests vs training | PSI > 0.2 | Investigate |
| Prediction drift | Mean predicted probability | ±20% from training mean | Investigate |
| Training-serving skew | Feature null rate in serving vs training | >2 percentage points | Fix pipeline |
| Error rate | HTTP 5xx / total requests | >1% over 1 hour | Rollback |
| Latency | P95 inference latency | >800 ms over 15 min | Investigate |

**Task 16.2 — Implement at least one**
Pick one signal and implement it as a script that queries `prediction_logs` and computes the metric. Run it against the last 24 hours of data.

**Deliverable:** the monitoring table plus a working script for one signal.
Cell 19 — Section 17: Rollback procedure
markdown
## Section 17 — Rollback Procedure
**Where this belongs:** a new section after Part 22 of blueprint 1.

**Why:** PDF Section 27 and Section 29 require an explicit rollback strategy. No blueprint addresses this.

**Task 17.1 — Define the rollback trigger**
Write the condition that triggers rollback. Example:
> *"If P95 latency > 1.5 s for 10 consecutive minutes, OR if HTTP 5xx rate > 2% for 5 minutes, OR if the model's live precision drops below [X] over a 500-prediction window."*

**Task 17.2 — Define the rollback mechanism**
- **Model rollback:** keep the previous `joblib` version in `models/archive/` and the previous MLflow registry version. Redeploy by pointing the app at the previous version.
- **Code rollback:** use `git revert` or redeploy the previous Docker image tag.
- **Database:** if schema changed, document the down-migration.

**Task 17.3 — Test the rollback**
- Deliberately deploy a broken model.
- Trigger the rollback condition.
- Confirm the system returns to the previous version.
- Record the time-to-recovery.

**Deliverable:** a written rollback runbook, plus a tested recovery time.
Cell 20 — Section 18: API performance metrics
markdown
## Section 18 — API Performance Measurement
**Where this belongs:** a new section after Section 12 above.

**Why:** Blueprint 3 §7 requires latency percentiles, throughput, and error rate. No blueprint measures them for the API.

**Task 18.1 — Load test**
Write a small script that sends 500 requests to the live `/predict/cancellation` endpoint. Record:
- Per-request latency.
- Success/failure flag.

**Task 18.2 — Report percentiles**
Compute and report:
- Mean latency
- P50
- P90
- P95
- P99
- Max
- Throughput (RPS)
- Error rate

**Task 18.3 — Cost per 1,000 requests**
Estimate the compute cost of 1,000 requests on your hosting tier. Report it.

**Deliverable:** a metrics table plus the load-test script.
Cell 21 — Section 19: Documentation enforcement
markdown
## Section 19 — Documentation Enforcement
**Where this belongs:** a final checklist at the end of the notebook.

**Why:** PDF Section 37 defines a documentation standard. Blueprint 1 only lists a README structure.

**Task 19.1 — Verify the repository contains**
- `README.md` with all 16 sections from PDF Section 37.
- `LICENSE`.
- `.gitignore` (excludes `.env`, `__pycache__`, `*.joblib` if not committed).
- `requirements.txt` or `pyproject.toml`.
- `src/`, `tests/`, `configs/`, `scripts/`, `notebooks/`, `docs/`.
- `docs/architecture.png` (or `.drawio`).
- `docs/postmortem_01.md`.

**Task 19.2 — Write the README sections**
Make sure these are present and non-empty:
1. Problem
2. Motivation
3. Architecture
4. Data
5. Setup
6. Usage
7. Method
8. Baseline
9. Experiments
10. Results
11. Error analysis
12. Deployment
13. Evaluation
14. Limitations
15. Security considerations
16. Future work

**Deliverable:** a screenshot of the repo tree plus a link to the completed README.
Cell 22 — Section 20: Final project scorecard
markdown
## Section 20 — Final Project Scorecard
**Where this belongs:** the last cell of the notebook.

**Why:** Blueprint 3 §11 defines the scorecard. It must be filled in for Project 1.

**Task 20.1 — Complete the scorecard**

- [ ] Business problem defined and stakeholders identified.
- [ ] Prediction contract written (target, horizon, action, costs).
- [ ] Baseline card recorded before any optimization.
- [ ] SQL feature query written, verified with `EXPLAIN`, saved to repo.
- [ ] Leakage audit documented at both SQL and model layer.
- [ ] At least three candidate models trained and compared.
- [ ] Calibration curve and Brier score computed before and after calibration.
- [ ] Temporal split compared to random split.
- [ ] Error slicing by group completed; underserved groups identified.
- [ ] Cost-driven threshold selected with real dollar values.
- [ ] MLflow tracking used; best model registered.
- [ ] Deliberate failure postmortem written and saved.
- [ ] Reproducibility manifest produced and verified.
- [ ] FastAPI implemented and tested.
- [ ] PostgreSQL prediction logging implemented and verified.
- [ ] Tests written and passing.
- [ ] Docker build works.
- [ ] CI/CD workflow runs green on push.
- [ ] Backend, database, and frontend deployed.
- [ ] Drift monitoring design written; at least one signal implemented.
- [ ] Rollback procedure written and tested.
- [ ] API performance metrics measured.
- [ ] README complete with all 16 sections.
- [ ] Architecture diagram present.

**Task 20.2 — Write the resume bullet**
Use the formula from blueprint 3 §12:

> *Action + System + Measurement + Business Impact*

Example structure (fill in with your real numbers):
> *"Engineered an order-cancellation risk classifier that improved recall from [X]% to [Y]% over a DummyClassifier baseline, reduced expected cost per 1,000 predictions by [Z]% via cost-driven threshold selection, and shipped as a Dockerised FastAPI service with PostgreSQL prediction logging and CI/CD."*

**Deliverable:** a fully checked scorecard plus a final resume bullet with real numbers.
Where to place this notebook in the repo
text
notebooks/
├── 01_data_understanding_and_eda.ipynb
├── 02_cancellation_model.ipynb
├── 03_quantity_model.ipynb
├── 04_model_validation.ipynb
└── 05_project1_completion.ipynb   ← this notebook